<img src="../../img/backdrop-wh.png" alt="Drawing" style="width: 300px;"/>

# TF-IDF

* * * 

<div class="alert alert-success">  
    
### Learning Objectives 
    
* Understand the TF-IDF algorithm, and how it's used in information retrieval.
* Understand the difference between TF-IDF and simple word counts.
* Use Scikit Learn to extract TF-IDF scores from our text data.
* Use TF-IDF scores to compare posts.
</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning:** Heads-up about tricky stuff or common mistakes.<br>
💭 **Reflection**: Reflecting on ethical implications, biases, and social impact in data science.<br>

### Sections
1. [Turning Words into Numbers](#words)
2. [Bag-Of-Words Representation](#bow)
3. [Testing TF-IDF with a toy dataset](#toy)
4. [Using TF-IDF on Reddit datasets](#reddit)
5. [Using TF-IDF to Find Similar Posts](#similar)

<a id='words'></a>

# Turning Words Into Numbers

In the previous notebook, we covered methods to clean and tokenize text into units (like words, bigrams, and trigrams).

For many data science applications, we need a way to convert the content of a text (the list of tokens) into useful numbers that can be used as the features of a model or analysis. One way to do this is to to use a method called Term Frequency-Inverse Document Frequency (TF-IDF). 

Even today, TF-IDF is:
- Used under the hood in many search engines and recommender systems;
- A baseline for evaluating more complex models like word embeddings and transformers;
- A great tool for small or interpretable projects where LLMs are overkill;

Note that turning words into numbers, ultimately, is a question of **representation**. A text representation is easier for humans to understand, but a numerical representation is easier to perform large scale computational analysis on. The more we change the representation, the more distant our reading of the text becomes. We may gain some benefits from this, but we must also keep in mind what we lose.

## Retrieving the Dataset


### Note on package installation

- If you are running this notebook on **DataHub**, you may need to **uncomment and run** the `%pip install ...` line below if you get an error about a missing package. Restart your kernel after running this cell!
- If you are working **locally** (on your own computer), you should already have all required packages installed via your Conda environment (see the ***"Local Python and Jupyter Setup"*** page on bCourses). Only use the `pip install` line if you see an ImportError and know what you’re doing.

In [ ]:
#%pip install scikit-learn gdown

> **Data transparency note**: All analyses in this notebook use text written by Reddit users on the r/AmItheAsshole subreddit (posts collected 2013–2022). The patterns surfaced by these methods reflect that community's discourse — not universal truths about language or morality. Keep this context in mind when interpreting results.

In [ ]:
# NOTE: There is no need to run the below `import gdown` cell, 
# which downloads the preprocesed data from Google Drive, 
# if you have preprocessed the data yourself, locally, using the 
# Week 1 `preprocessing_python` notebook.

import gdown

file_id = "1rAnEFVYAMu_DVcZNK3pl7WI4meeMvVVg"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "../../data/aita_pp.csv", quiet=False)

In [ ]:
import os
import pandas as pd

In [ ]:
df = pd.read_csv('../../data/aita_pp.csv')

**Use the `2_TF_IDF_Project.ipynb` notebook to run the TF-IDF operations explained in this notebook on your own data.**

<a id='bow'></a>

# Bag-Of-Words Representation

Before we do TF-IDF, we need to learn something about **bag-of-words (BOW)** models.

The idea of BOW models is to encode the corpus in terms of word frequencies. In a bag-of-words, we taken some text, tokenize it, and then tabulate the frequencies of each token. The numerical representation of the text, then, is a vector indicating the frequencies of each token for that text.

For example, if we're considering a Reddit post on r/amitheasshole: 

<img src="../../img/bow.svg"  width="600" height="300">


We take each token from the review, "toss it in a bag", and count up the frequencies of each word. The numerical representation, then, is the vector on the right: the number of appearances of each token. The "bag" here denotes that we are not modeling structure within the text - only the frequencies of the words.


### Document Term Matrix

In most text corpora, we will have many samples or *documents*. For example, in our Reddit dataset, we have many posts. Each post stands on its own as a unique sample: it can be thought of as a unique document in the entire *corpus*. 

Since they are all related to each other, many tokens might be shared across posts. So, when creating the bag-of-words model, we can tokenize across all documents, forming a *vocabulary*. Then, we can represent a single document by which of the tokens in the vocabulary are represented, and their frequency within the document.

If the vocabulary has $V$ tokens, then each document will be encoded in a $V$-dimensional vector. If there are $D$ documents, the entire dataset can be represented in a $D \times V$ matrix, where each row corresponds to the document, and each column corresponds to the token (or "term"). This $D \times V$ matrix is a **document term matrix** (DTM).

Let's consider a simple example. Suppose we have the "documents":

```
["You are at a workshop. Are you ready?",
 "Welcome to Berkeley!",
 "I am teaching a workshop."]
```

The unique (word) tokens in this "corpus", in alphabetical order, are:

```
[a, am, are, at, berkeley, i, ready, teaching, to, welcome, workshop, you]
```

The DTM can be formed by going through each document, ticking off the frequency of each token in each document, and plugging this number into the matrix:

$$
\begin{array}{c|cccccccccccc}
 & \text{a} & \text{am} & \text{are} & \text{at} & \text{berkeley} & \text{i} & \text{ready} & \text{teaching} & \text{to} & \text{welcome} & \text{workshop} & \text{you} \\\hline
\text{Document 1} & 1 & 0 & 2 & 1 & 0 & 0 & 1 & 0 & 0 & 0 & 1 & 1 \\
\text{Document 2} & 0 & 0 & 0 & 0 & 1 & 0 & 0 & 0 & 1 & 1 & 0 & 0 \\
\text{Document 3} & 1 & 1 & 0 & 0 & 0 & 1 & 0 & 1 & 0 & 0 & 1 & 0 \\
\end{array}
$$

The numerical representation for each document is a row in the matrix. For example, Document 1 has numerical representation $[1, 0, 2, 1, 0, 0, 1, 0, 0, 0, 1, 1]$.

To create a DTM, we will use the `CountVectorizer` from the package `sklearn`, a heavily used machine learning package.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

Here is the workflow:

1. We first create a `CountVectorizer` object, and choose specific settings for how we'll go about creating the DTM. Check out the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) to see what options are available.
2. Then, we "fit" this `CountVectorizer` object to the data. In this context, "fitting" consists of establishing a vocabulary of tokens from the documents in your dataset.
3. Finally, we "transform" the data according to the "fitted" `CountVectorizer` object. This means taking our text data and transforming it into a DTM according to the vocabulary established by the "fitting" step.
4. You can do steps 2 and 3 in one fell swoop using a `fit_transform` function.


In [ ]:
corpus = [
  'My cat has paws.',
  'Can we let the dog out?',
  'Our dog really likes the cat but the cat does not agree.']
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)
test_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
test_df

Each column in the matrix represents a unique word in the vocabulary, while each row represents the document in our dataset. In this case, we have three sentences (i.e. the document), and therefore we three rows. The values in each cell are the word counts. Note that with this representation, counts of some words could be 0 if the word did not appear in the corresponding document. 


🔔 **Question**: Lets look at the shape of the matrix. How many unique words are there in the matrix?'

In [ ]:
X.shape

Now, we have numbers representing the contents of the documents! Matrices like this are the simplest way to represent texts. However, it biases most frequent words and ends up ignoring rare words which could have helped is in processing our data more efficiently.

Often, we not only want to focus on the frequency of words present in the corpus but also want to know the importance of the words. This is where TF-IDF (term frequency-inverse document frequency) comes in.

# Implementing TF-IDF

TF-IDF, short for **term frequency–inverse document frequency**, is a metric that reflects how important a word is to a **document** in a collection or **corpus**. When talking about text datasets, the dataset is called a corpus, and each datapoint is a document. A document can be a post, a paragraph, a webpage, whatever is considered the individual unit of text for a given datset. A **term** is each unique token in a document (we previously also referred to this as **type**). 

For example in a corpus of sentences, a document might be: `"I went to New York City in New York state."` 

The processed tokens in that document might be: `[went, new_york, city, new_york, state]`.

The document would have four unique terms: `[went, new_york, city, state]`.

The TF-IDF value increases proportionally to the number of times a word appears in the document (the term frequency, or TF), and is offset by the number of documents in the corpus that contain the word (the inverse document frequency, or IDF). This helps to adjust for the fact that some words appear more frequently in general – such as articles and prepositions.

We won't go into much detail about the math behind calculating the TF-IDF (see the D-Lab Text Analysis workshop videos to see more). The key components to remember are:

1. There is one TF-IDF score per unique word and unique document.
2. A high TF-IDF score suggests that word is descriptive of that document.
3. A low TF-IDF score may be because either the word is not frequent in that document, or that it is frequent in many documents in the dataset - either way, it may not be a good descriptor of that document.

The intuition is that if a word occurs many times in one post but rarely in the rest of the corpus, it is probably useful for characterizing that post; conversely, if a word occurs frequently in a post but also occurs frequently in the corpus, it is probably less characteristic of that post.

## The TF-IDF Formula: A Worked Example

Let's make the math concrete before we use sklearn to do it for us.

**Term Frequency (TF)** for term $t$ in document $d$:

$$\text{TF}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total tokens in } d}$$

**Inverse Document Frequency (IDF)** for term $t$ across corpus of $N$ documents:

$$\text{IDF}(t) = \log\left(\frac{N + 1}{df(t) + 1}\right) + 1$$

where $df(t)$ is the number of documents containing $t$. The "+1" smoothing (enabled by `smooth_idf=True`) prevents zero-division for terms that appear in every document.

**TF-IDF** is simply:

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

**Example**: Imagine a corpus of 5 posts. The word "husband" appears 4 times in post #2 (which has 100 tokens total), and appears in 2 of the 5 posts.

- TF = 4/100 = 0.04  
- IDF = log((5+1)/(2+1)) + 1 ≈ log(2) + 1 ≈ 1.69  
- TF-IDF ≈ 0.04 × 1.69 ≈ **0.068**

A common word like "said" that appears in every post would have IDF ≈ 1.0, pulling its TF-IDF score down even if frequent within one post. This is how TF-IDF suppresses noise while surfacing distinctive terms.

⚠️ **Warning:**  Word order is still not retained in this type of featurization, since all that is counted is overall frequency of a word in a document. So it is still a **bag-of-words** approach.

<a id='toy'></a>

# Testing TF-IDF with a Toy Dataset

Let's try TF-IDF out with a toy dataset. Here we have three documents about Python, but with different meanings. If we are trying to distinguish between these documents, the word "Python" would not be very useful, since it occurs in all of the documents, but other terms might, like "Monty", "snake", etc.

In [ ]:
document1 = """Python is a 2000 made-for-TV horror movie directed by Richard
Clabaugh. The film features several cult favorite actors, including William
Zabka of The Karate Kid fame, Wil Wheaton, Casper Van Dien, Jenny McCarthy,
Keith Coogan, Robert Englund (best known for his role as Freddy Krueger in the
A Nightmare on Elm Street series of films), Dana Barron, David Bowe, and Sean
Whalen."""

document2 = """Python, from the Greek word (πύθων/πύθωνας), is a genus of
nonvenomous pythons[2] found in Africa and Asia. Currently, 7 species are
recognised.[2] A member of this genus, P. reticulatus, is among the longest
snakes known."""

document3 = """Monty Python (also collectively known as the Pythons) are a British 
surreal comedy group who created the sketch comedy television show Monty Python's 
Flying Circus, which first aired on the BBC in 1969. Forty-five episodes were made 
over four series."""

document4 = """Python is an interpreted, high-level, general-purpose programming language. 
Created by Guido van Rossum and first released in 1991, Python's design philosophy emphasizes 
code readability with its notable use of significant whitespace. Its language constructs and 
object-oriented approach aim to help programmers write clear, logical code for small and 
large-scale projects."""

document5 = """The Colt Python is a .357 Magnum caliber revolver formerly
manufactured by Colt's Manufacturing Company of Hartford, Connecticut.
It is sometimes referred to as a "Combat Magnum". It was first introduced
in 1955, the same year as Smith &amp; Wesson's M29 .44 Magnum. The now discontinued
Colt Python targeted the premium revolver market segment."""

document6 = """The Pythonidae, commonly known simply as pythons, from the Greek word python 
(πυθων), are a family of nonvenomous snakes found in Africa, Asia, and Australia. 
Among its members are some of the largest snakes in the world. Eight genera and 31
species are currently recognized."""

test_list = [document1, document2, document3, document4, document5, document6]

## Using `CountVectorizer()`

Let's use `CountVectorizer()` again, and customize it a bit. 

We will set the variable `cv` to an instance of `CountVectorizer()`, and change two parameters:
    - Set `max_df` (max document frequency) to get rid of words that appear in more than 85% of the corpus. 
    - Set `stop_words` to include the English stopword list, in order to leave those stopwords out of the calculations as well.
    
Check the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) if you need help on how to change these parameters!

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_df=0.85, stop_words='english')

Now let's run `fit_transform()` on our `test_list` with these settings. We'll also create a dataframe with all the word counts.

In [ ]:
word_count_vector = cv.fit_transform(test_list)
pd.DataFrame(word_count_vector.toarray(), columns=cv.get_feature_names_out())

🔔 **Question**: How many documents and unique words are in this dataset?

## Using `TfidfTransformer`

Next, we need to compute the inverse document frequency values. We'll call [`tfidf_transformer.fit()`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html) on the word counts we computed earlier.


In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf_transformer = TfidfTransformer() 
tfidf_transformer.fit(word_count_vector)

To get a glimpse of how the IDF values look, let's put these into a DataFrame and sort by weights. Remember, a low IDF indicates something that is less unique.

In [ ]:
# Print IDF values 
df_idf = pd.DataFrame(tfidf_transformer.idf_, index=cv.get_feature_names_out(), columns=["idf_weights"]) 
# Sort ascending 
df_idf.sort_values(by=['idf_weights'])

Notice that the words "python" and "in" have the lowest IDF values. This is expected: these words appear in each and every document in our collection. The lower the IDF value of a word, the less unique it is to any particular document.

Now that we have the idf values, we can compute the TF-IDF scores for our set of documents using `.transform()`

In [ ]:
tf_idf_vector = tfidf_transformer.transform(word_count_vector)

By invoking `tfidf_transformer.transform()` we are computing the TF-IDF scores for our docs. Internally, this is weighting TF scores by their IDF scores, so that the more unique a word, the more its frequency counts in a given document.

Let’s print the TF-IDF values of the first document to see if it makes sense. We place the TF-IDF scores from the third document into a `pandas` data frame and sort it in descending order of scores. We can replace the index in `tf_idf_vector[]` to select a different document to check.

In [ ]:
feature_names = cv.get_feature_names_out() 
  
# Print the scores 
df_test = pd.DataFrame(tf_idf_vector[2].T.todense(), index=feature_names, columns=["tfidf"]) 
df_test.sort_values(by=["tfidf"], ascending=False)

What are the most distinctive words for document 3 (Highest TF-IDF scores?) Does it made sense given the document and corpus?

from sklearn.feature_extraction.text import TfidfVectorizer

# TfidfVectorizer parameter guide:
#   max_df=0.85    — ignore terms appearing in more than 85% of documents (corpus-wide noise like "aita")
#   max_features=1000 — keep only the top 1000 terms by corpus frequency (limits matrix size)
#   min_df=5       — ignore terms appearing in fewer than 5 documents (rare noise/typos)
#   decode_error='ignore' — silently drop any bytes that can't be decoded (handles encoding artifacts)
#   smooth_idf=True — add 1 to numerator/denominator in IDF formula to avoid zero-division
#   use_idf=True   — multiply TF by IDF; setting False gives raw TF counts instead
tfidf_vectorizer = TfidfVectorizer(max_df=0.85,
                                   max_features=1000,
                                   min_df=5,
                                   decode_error='ignore',
                                   stop_words='english',
                                   smooth_idf=True,
                                   use_idf=True)

# Fit and transform the texts
tfidf = tfidf_vectorizer.fit_transform(df['pp_text'])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Settings that you use for count vectorizer will go here
tfidf_vectorizer = TfidfVectorizer(max_df=0.85,
                                   max_features=1000,
                                   decode_error='ignore',
                                   stop_words='english',
                                   smooth_idf=True,
                                   use_idf=True)

# Fit and transform the texts
tfidf = tfidf_vectorizer.fit_transform(df['pp_text'])


The `tfidf` object we created is a so-called **sparse matrix**. We will first need to convert this into something we can read and make use of.

In [ ]:
# "sparse matrix"
tfidf

We can put this sparse matrix into a DataFrame. We will create a DF from the sparse dataset.

In [ ]:
# Place TF-IDF values in a DataFrame
feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame.sparse.from_spmatrix(tfidf, columns=feature_names)

Let's have a look:

In [ ]:
tfidf_df.iloc[:5, :20]

Note that this `tfidf_idf` is a lot like a Document-Term Matrix, except with TF-IDF counts! The columns represent every term in our 1000-word vocabulary. The rows represent documents in which these words appear. 

Note that most values are 0. This is because most words in our vocabulary actually don't show up in most documents.

We can retrieve the highest TF-IDF values across documents in this DataFrame by just summing all TF-IDF values, and then calling `.sort_values()` on our DataFrame.

In [ ]:
# Highest TF-IDF values across documents
tfidf_df.sum().sort_values(ascending=False)

## Top TF-IDF Terms per Post

After computing the TF-IDF matrix, we can extract the top-scoring terms for a specific post (post 10 below). This helps us understand what TF-IDF actually surfaces and why it’s different from raw frequency.

In [ ]:
import numpy as np

def get_top_tfidf_words(row, features, top_n=10):
    top_indices = np.argsort(row)[::-1][:top_n]
    return [(features[i], row[i]) for i in top_indices]

# Example: document 10
top_words = get_top_tfidf_words(tfidf[10].toarray()[0], tfidf_vectorizer.get_feature_names_out())
for word, score in top_words:
    print(f"{word}: {score:.4f}")

Let's look at the post itself to see what terms TF-IDF is considering "distinctive".

In [ ]:
df.selftext[10]

We can visualize these TF-IDF-weighted terms as well. This code saves the plot in a PNG file.

In [ ]:
import matplotlib.pyplot as plt

def plot_top_terms(tfidf_vector, feature_names, doc_id=0, top_n=10):
    row = tfidf_vector[doc_id].toarray()[0]
    top_indices = row.argsort()[-top_n:][::-1]
    terms = [feature_names[i] for i in top_indices]
    scores = [row[i] for i in top_indices]

    plt.figure(figsize=(8, 5))
    plt.barh(terms[::-1], scores[::-1])
    plt.title(f"Top {top_n} TF-IDF Terms for Document {doc_id}")
    plt.xlabel("TF-IDF Score")
    plt.tight_layout()
    plt.savefig(f"outputs_lesson/top_terms_doc_{doc_id}.png", dpi=300)
    plt.show()

# Change doc_id below to get data for a different post
plot_top_terms(tfidf, tfidf_vectorizer.get_feature_names_out(), doc_id=10)

## Top Terms Across the Corpus (Mean TF-IDF)

Now, let's move from document-level to corpus-level views:

In [ ]:
from sklearn.decomposition import PCA

# Reduce to 2 components for visualization
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(tfidf.toarray())

# Map flair to colors (using only the two main verdicts)
flair_map = {
    'Not the A-hole': 'steelblue',
    'Asshole': 'tomato',
}
colors = df['flair_text'].map(flair_map).fillna('lightgrey')

plt.figure(figsize=(9, 6))
for label, color in flair_map.items():
    mask = df['flair_text'] == label
    plt.scatter(coords[mask, 0], coords[mask, 1],
                c=color, label=label, alpha=0.3, s=5)
plt.scatter(coords[colors == 'lightgrey', 0], coords[colors == 'lightgrey', 1],
            c='lightgrey', label='Other', alpha=0.1, s=3)

plt.title("TF-IDF Document Space (PCA projection)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.legend(markerscale=4)
plt.tight_layout()
plt.savefig("outputs_lesson/pca_tfidf.png", dpi=300)
plt.show()

print(f"Total variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%")

## Visualizing Document Space with PCA

Each document is now a 1000-dimensional TF-IDF vector. We can't visualize 1000 dimensions directly, but **Principal Component Analysis (PCA)** can project that space down to 2 dimensions while preserving as much variance as possible.

Coloring by flair lets us ask: do NTA and YTA posts cluster separately in TF-IDF space? If they do, that suggests their vocabulary is systematically different — which would make TF-IDF a meaningful signal of community judgment.

⚠️ **Warning**: PCA is a linear projection. It may not reveal all meaningful structure; t-SNE or UMAP (used in later weeks) can capture non-linear patterns that PCA misses.

In [ ]:
mean_tfidf = tfidf.mean(axis=0).A1
terms = tfidf_vectorizer.get_feature_names_out()
top_indices = mean_tfidf.argsort()[-10:][::-1]

top_terms = [terms[i] for i in top_indices]
top_scores = [mean_tfidf[i] for i in top_indices]

plt.figure(figsize=(8, 5))
plt.barh(top_terms[::-1], top_scores[::-1])
plt.title("Top TF-IDF Terms Across Corpus")
plt.xlabel("Mean TF-IDF Score")
plt.tight_layout()
plt.savefig("outputs_lesson/top_terms_corpus.png", dpi=300)
plt.show()

## 💭 Reflection
- What kinds of words does TF-IDF seem to prioritize, and which does it ignore? Are these terms really the most "important" in a post?  
- How might this weighting reinforce certain biases (e.g. technical terms, rare slang, moral judgments)?  
- What assumptions are we making about what matters in language?

<a id='similar'></a>

## Using TF-IDF to Find Similar Posts

We can use TF-IDF to work out the similarity between any pair of documents. So given one post or comment, we could see which posts or comments are most similar. This can be useful if you're trying to find other examples of a pattern you have found and want to explore further.

Let's choose a particular document, and try and find similar documents.

In [ ]:
doc_idx = 25

In [ ]:
df['selftext'].iloc[doc_idx]

🔔 **Question:** What is this post about?

Let's have a quick look at the TF-IDF scores for the words in this submission to see if these words are indeed typical for this particular submission. 

In [ ]:
tfidf_df.loc[doc_idx].sort_values(ascending=False)

🔔 **Question:** Do the distinctive words have to do with the topic of the post?

## Calculating similarity 

Now let's find the closest posts to this one. The fact that our documents are now in a vector space allows us to make use of mathematical similarity metrics.

**Cosine similarity** is one metric used to measure how similar the documents are irrespective of their size. Mathematically, it measures the cosine of the angle between two vectors projected in a multi-dimensional space. It is equal to 1 if the documents are the same, and decreases to 0 the more dissimilar they are.

We can use a cosine similarity function from `sklearn` to calculate the cosine similarity between each pair of documents:

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(tfidf)
similarities.shape

We can put the text and scores in a dataframe, and sort by the score:

In [ ]:
similar_df = pd.DataFrame({
    'text': df['selftext'].values,
    'score': similarities[doc_idx]}).sort_values('score', ascending=False)

The top document will be the document itself (it's going to have a similarity of 1 with itself). So we look at the next document.

In [ ]:
similar_df['text'].iloc[0]

In [ ]:
similar_df['text'].iloc[1]

💭 **Reflection**: Think about why this document seems similar, in terms of TF-IDF scores, to the one we pulled out first. Do you agree with this calculation of similarity? 


## Using TF-IDF to Find Posts

TF-IDF has been used in search algorithms. This makes sense: after all, it can tell you which words are uncommonly frequent in some text. In this sense, TF-IDF can act as a keyword or topic identifier.

For instance, if we would want to look for a text in our DF that has a high TF-IDF score for the word "husband", we could create a **boolean mask** of our `tfidf_df`. Below, we only select those rows where the column "husband" has a TF-IDF score higher than `.5`. We can then use that mask to subset our original `df`! This is because the rows in both DataFrames refer to the same thing: our Reddit posts.

In [ ]:
# Subsetting one DF with the mask of another DF
tfidf_husband_df = df[tfidf_df['husband'] > .5]
tfidf_husband_df.head(3)

We can now get the "selftext" column from this new subsetted DataFrame, and then get the first post just to have a look:

In [ ]:
print(tfidf_husband_df['selftext'].iloc[0])

## Using TF-IDF Correlations to Explore Biases

Calculating correlations of TF-IDF values can be a useful technique to explore relationships between words or terms in a corpus. By analyzing the correlations, we can identify whether certain words tend to appear together more frequently or if they are related to similar topics. 

Pandas allows us to trace pairwise correlations in a DataFrame using the `corr()` method. Note that this creates a new DataFrame that is square and symmetric. Each element in the resulting DataFrame represents the correlation coefficient between all terms.

In [ ]:
corr = tfidf_df.corr()

In [ ]:
## 💭 Reflection: What Is TF-IDF Blind To?

TF-IDF treats text as a **bag of words** — a collection of tokens with no order, no context, and no syntax. It can tell you which words are statistically distinctive, but it cannot tell you:

- **Negation**: "I did *not* take the money" and "I did take the money" look nearly identical to TF-IDF.
- **Sarcasm and irony**: Common in online communities like r/AmITheAsshole. A word like "obviously" has a very different meaning depending on tone.
- **Context of co-occurrence**: Whether "angry" appears near "husband" or near "myself" changes meaning completely — but TF-IDF can't see that.
- **Who is speaking**: The poster's identity, power position, or relationship to the people they describe shapes the story. TF-IDF counts words regardless of whose perspective they come from.

Looking at the NTA vs YTA comparison you just made:

🔔 **Question**: Do the different vocabulary profiles reflect a real difference in how people *narrate* being in the right vs being in the wrong? Or could they reflect a different kind of situation being described?

🔔 **Question**: Imagine you trained a classifier to predict NTA vs YTA from TF-IDF features. What would it be learning, exactly? What would it not be learning?

In [ ]:
import numpy as np

# Subset posts by verdict (normalize flair labels — they vary in this dataset)
nta_mask = df['flair_text'].str.lower().str.contains('not the a', na=False)
yta_mask = df['flair_text'].str.lower().str.contains('asshole', na=False) & ~nta_mask

nta_tfidf = tfidf[nta_mask.values]
yta_tfidf = tfidf[yta_mask.values]

# Mean TF-IDF per group
nta_mean = nta_tfidf.mean(axis=0).A1
yta_mean = yta_tfidf.mean(axis=0).A1

features = tfidf_vectorizer.get_feature_names_out()

# Top 15 distinctive words per group (high mean TF-IDF)
top_n = 15
nta_top_idx = nta_mean.argsort()[-top_n:][::-1]
yta_top_idx = yta_mean.argsort()[-top_n:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh([features[i] for i in nta_top_idx[::-1]],
             [nta_mean[i] for i in nta_top_idx[::-1]], color='steelblue')
axes[0].set_title("Top Terms: Not the A-hole (NTA)")
axes[0].set_xlabel("Mean TF-IDF")

axes[1].barh([features[i] for i in yta_top_idx[::-1]],
             [yta_mean[i] for i in yta_top_idx[::-1]], color='tomato')
axes[1].set_title("Top Terms: Asshole (YTA)")
axes[1].set_xlabel("Mean TF-IDF")

plt.tight_layout()
plt.savefig("outputs_lesson/nta_vs_yta_tfidf.png", dpi=300)
plt.show()

print(f"NTA posts: {nta_mask.sum()}  |  YTA posts: {yta_mask.sum()}")

## Group Comparison: NTA vs YTA Vocabulary

One of the most powerful uses of TF-IDF is comparing the characteristic vocabulary of two groups. Here, we ask: **which words are most distinctive of posts judged "Not the A-hole" vs "Asshole"?**

This is not just a technical exercise. These two verdicts encode a community's moral judgment about social behavior. If their vocabulary differs systematically, that tells us something about how people narrate being in the right vs being in the wrong.

🔔 **Question**: Before running the code, make a prediction. What kinds of words do you expect to be distinctive for NTA posts? For YTA posts?

We can use this new DataFrame to compare two concepts in our data. We will look for the columns of "husband" and "wife" – two concepts we expect to appear in this dataset, and that we could imagine being related to gender bias. We then sort the values of the resulting DataFrame based on the "wife" columns in descending order, and print the first 20 values. 

These will be the top-20 words that are most strongly correlated to the term "wife". 

In [ ]:
corr[['husband','wife']].sort_values(by='wife',ascending=False)[:30]

## 💭 Reflection: 

- Do these related terms make sense? 
- Do you see some terms that could be indicative of a bias towards women in the data? 
- What happens if you change the `by=` sortation to `men`? What words appear now, and how do they make sense?

<div class="alert alert-success">

## ❗ Key Points

* Term Frequency, or TF, reflects how often a unique token appears in a corpus.
* Inverse Document Frequency (IDF) reflects the number of documents in the corpus that contain a term. 
* The TF-IDF score of a word reflects how important that word is to a document in a collection or corpus.
* Methods from `scikit-learn` can be used to generate TF and TF-IDF scores for a given corpus.
* TF-IDF scores can be used to calculate how similar documents are. We can do this with mathematical similarity metrics, such as cosine similarity.
    
</div>